In [1]:
import rasterio as rio
import pickle as pkl
import pandas as pd
import geopandas as gpd
import os
from rasterio.enums import Resampling
import numpy as np
from pylab import *
%matplotlib inline
from data_extraction import *

In [2]:
def save_resampled(in_raster, crs, transform, outfile):
    with rio.open(
            outfile,
            mode="w",
            driver="GTiff",
            height=in_raster.shape[1],
            width=in_raster.shape[2],
            count=in_raster.shape[0],
            dtype=in_raster.dtype,
            crs=crs,
            transform=transform,
    ) as new_dataset:
            new_dataset.write(in_raster, )

In [5]:
# trained generic model
#exp_dir = f'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/results/2b_classifier_performance/RF_generic/'
clf_dir = f'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/code/lichen_git/lichen/classifier_lib/generic_RF/'
clf_name = 'RF_generic'
with open(clf_dir+f'{clf_name}.pkl', 'rb') as file:      
    clf_gen = pkl.load(file)

### Get features and true labels
- load rgb, chm and label shape 
- label shape rasterize
- resample both to spatial resolutions (1, 2, 4, 8, 16, 32 cm) using Resampling=med for rgb and Resampling=mode for labels
- extract features and class labels to df
- test sites are: 'CG1-8B', 'F3-20B', CS-103A', 'ZF20-11A'

In [6]:
data_dir = data_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/2_labelling/hp'
site = 'CG1-8B'
rgb_file = os.path.join(data_dir, site, f'{site}_hp_transparent_mosaic_group1.tif')
#chm_file = os.path.join(data_dir, site, 'dem_and_derivatives', f'{site}_hp_chm_stratified.tif')
chm_shp_file = os.path.join(data_dir, site, 'stratified_sampling', f'{site}_hp_chm_stratified.shp')
label_file = os.path.join(data_dir, site, f'{site}_hp_labelledpoints.gpkg')

### resample rgb dataset to coarser resolution and save new tif
resampled data is stored in 'data' array

In [14]:
new_rgb = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/paper/reprojections/CG1-8B/CG1-8B_16cm.tif'

# resample
upscale_factor = 0.5
with rio.open(new_rgb) as dataset:

    # resample data to target shape
    data = dataset.read(
        out_shape=(
            dataset.count,
            int(dataset.height * upscale_factor),
            int(dataset.width * upscale_factor)
        ),
        resampling=Resampling.average
    )

    # scale image transform
    transform = dataset.transform * dataset.transform.scale(
        (dataset.width / data.shape[-1]),
        (dataset.height / data.shape[-2])
    )

# saving resampled data to new tif
results_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/paper/reprojections/CG1-8B'

out1 = f'{results_dir}/{site}_32cm.tif'
crs = dataset.crs
transform = transform
#save_meltdate_tif(data, crs, transform, out1)

In [17]:
# This was the most straightforward solution I could think of. load_site_data() is the function living in data_extraction.py 
# and which we have also been using to create training data for the 0.5 cm res data.

data_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/2_labelling/hp'
site = 'CG1-8B'
combined_raster = load_site_data(data_dir, site)
class_map = {
    'lichen': 1,
    'rock': 4,
    'broadleaf': 5,
    'needleleaf': 6,
    'deadwood': 7,
    'graminoids': 8,
    'moss': 9,
    'soil': 10,
    'low_green': 12,
    'dry_branches': 13,
}
df, _ = convert_to_dataframe(combined_raster, site, class_map=class_map)


results_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/paper/reprojections/CG1-8B'
out1 = f'{results_dir}/{site}_combined.tif'
crs = dataset.crs
transform = dataset.transform
save_resampled(combined_raster, crs, transform, out1)

### Troubleshooting

In [24]:
rgb_file = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/paper/reprojections/CG1-8B/CG1-8B_16cm.tif'
with rio.open(rgb_file) as f:
        # Get raster metadata
        height = f.height
        width = f.width
        crs = f.crs
        transform = f.transform
        output_raster = np.ndarray((6,height,width), dtype=np.uint8)
        rgb_raster = f.read((1,2,3), out=output_raster[1:4,:,:])

# shapefiles
chm_shp_file = os.path.join(data_dir, site, 'stratified_sampling', f'{site}_hp_chm_stratified.shp')
label_file = os.path.join(data_dir, site, f'{site}_hp_labelledpoints.gpkg')

label_gdf = gpd.read_file(label_file).to_crs(crs).dropna(subset=['class', 'certainty'])
label_gdf['certainty'] = label_gdf['certainty'].astype(np.uint8)
chm_gdf = gpd.read_file(chm_shp_file).to_crs(crs)

In [29]:
combined_gdf = process_overlapping_shapes(label_gdf, chm_gdf)

In [32]:
test = rasterize(
            ((r['geometry'], r['class']) for _, r in combined_gdf.iterrows()),
            out_shape=(height, width),
            out=output_raster[0,:,:],
            transform=transform
        )

In [38]:
for _, r in combined_gdf.iterrows():
    print((r['geometry'], r['class']))
print(transform)
#486481.6305400000419468,6761539.3066900009289384 : 
#486533.0324400000390597,6761617.8451900007203221

(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 6)
(<POLYGON ((4.86e+05 6.76e+06, 4.86e+05 6.76e+06, 4.86e+05 6.76e+06, 4.86e+05...>, 6)
(<POLYGON ((4.86e+05 6.76e+06, 4.86e+05 6.76e+06, 4.86e+05 6.76e+06, 4.86e+05...>, 6)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 6)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 13)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 13)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 6)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 6)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 6)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 6)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.87e+05...>, 6)
(<POLYGON ((4.87e+05 6.76e+06, 4.87e+05 6.76e+06, 4.

In [29]:
import osgeo.gdal
osgeo.gdal.VersionInfo()
print (osgeo.gdal.__version__)

3.6.2
